In [10]:
import duckdb
import time

# =========================
#  CONFIG – EDIT THESE
# =========================
WX_FILE  = "/workspace/data/pa_subsets/pa_z_18mr25.parquet"       # wxareas input (GeoParquet, Parquet, etc.)
ZIP_FILE = "/workspace/data/pa_subsets/pa_zcta_tl_2020_us_zcta520.parquet"      # zipcodes input
WX_GEOM_COL  = "geometry"                   # geometry column name in wxareas
ZIP_GEOM_COL = "geometry"                   # geometry column name in zipcodes

OUTPUT_CSV = "/workspace/data/duckdb_geojoin_results.csv"

# Choose an equal-area projection for area in m² (5070 = NAD83 / Conus Albers)
TARGET_SRID = 5070

# =========================
#  CONNECT & LOAD SPATIAL
# =========================
con = duckdb.connect("geojoin_duckdb.db")

con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

# =========================
#  LOAD INPUT INTO TABLES
# =========================
# If your files are Parquet/GeoParquet:
con.execute(f"""
    CREATE OR REPLACE TABLE wxareas_raw AS
    SELECT * FROM read_parquet('{WX_FILE}');
""")

con.execute(f"""
    CREATE OR REPLACE TABLE zipcodes_raw AS
    SELECT * FROM read_parquet('{ZIP_FILE}');
""")

# If instead you have shapefiles, you can use:
# con.execute(\"\"\"
#     CREATE OR REPLACE TABLE wxareas_raw AS
#     SELECT * FROM ST_Read('path/to/wxareas.shp');
# \"\"\")
# con.execute(\"\"\"
#     CREATE OR REPLACE TABLE zipcodes_raw AS
#     SELECT * FROM ST_Read('path/to/zipcodes.shp');
# \"\"\")

# =========================
#  PRINT ROW COUNTS
# =========================
wx_count = con.execute("SELECT COUNT(*) FROM wxareas_raw;").fetchone()[0]
zip_count = con.execute("SELECT COUNT(*) FROM zipcodes_raw;").fetchone()[0]

print(f"wxareas rows:  {wx_count}")
print(f"zipcodes rows: {zip_count}")

# =========================
#  CHECK / SET SRID (OPTIONAL)
# =========================
# Peek at SRIDs (uncomment to inspect)
# print(con.execute(f"SELECT DISTINCT ST_SRID({WX_GEOM_COL}) FROM wxareas_raw LIMIT 10;").fetchall())
# print(con.execute(f"SELECT DISTINCT ST_SRID({ZIP_GEOM_COL}) FROM zipcodes_raw LIMIT 10;").fetchall())

# For this example, we’ll assume they’re currently EPSG:4326 and transform to TARGET_SRID.
# Adjust if your data is already projected.

SRC_CRS = 'EPSG:4326'   # assuming input is lon/lat
DST_CRS = 'EPSG:5070'   # equal-area for CONUS

con.execute(f"""
    CREATE OR REPLACE TABLE wxareas AS
    SELECT
        *,
        ST_Transform(geometry, '{SRC_CRS}', '{DST_CRS}') AS geom_aea
    FROM wxareas_raw;
""")

con.execute(f"""
    CREATE OR REPLACE TABLE zipcodes AS
    SELECT
        *,
        ST_Transform(geometry, '{SRC_CRS}', '{DST_CRS}') AS geom_aea
    FROM zipcodes_raw;
""")


# =========================
#  SPATIAL INDEXES
# =========================
# DuckDB spatial supports RTREE indexes
con.execute("CREATE INDEX IF NOT EXISTS idx_zip_geom_aea ON zipcodes USING RTREE(geom_aea);")
con.execute("CREATE INDEX IF NOT EXISTS idx_wx_geom_aea ON wxareas USING RTREE(geom_aea);")

# =========================
#  GEOJOIN QUERY
# =========================
geojoin_sql = """
    SELECT
        z.ZCTA5CE20 AS zipcode,       -- ZIP code
        z.GEOID20   AS GEOID20,       -- GEOID
        w.NAME      AS weather_name,  -- wx area name
        w.CWA       AS cwa,           -- CWA
        ST_Area(ST_Intersection(z.geom_aea, w.geom_aea)) AS area_m2
    FROM zipcodes z
    JOIN wxareas w
      ON ST_Intersects(z.geom_aea, w.geom_aea)
"""

# =========================
#  TIMED RUNNER
# =========================
def timed_geojoin(run_type: str) -> float:
    t0 = time.perf_counter()

    if run_type == "cold":
        # Run full query once to populate caches
        _ = con.execute(geojoin_sql).fetchall()

    elif run_type == "warm":
        # Run again with warmed pages/cache
        _ = con.execute(geojoin_sql).fetchall()

    t1 = time.perf_counter()
    return t1 - t0



# =========================
#  COLD RUN (WITH CSV)
# =========================
cold_time = timed_geojoin("cold")
print(f"Cold join compute only: {cold_time:.3f} seconds")



# =========================
#  HOT RUN (CACHE WARMED)
# =========================
warm_time = timed_geojoin("warm")
print(f"Warm join compute only: {warm_time:.3f} seconds")

# Count successful joins
successful_joins = con.execute(f"SELECT COUNT(*) FROM ({geojoin_sql})").fetchone()[0]
print(f"Successful joins: {successful_joins}")
print("Writing CSV…")
con.execute(f"""
    COPY ({geojoin_sql})
    TO '{OUTPUT_CSV}'
    (HEADER, DELIMITER ',');
""")
print(f"CSV written to: {OUTPUT_CSV}")

wxareas rows:  78
zipcodes rows: 1833
Cold join compute only: 3.446 seconds
Warm join compute only: 2.502 seconds
Successful joins: 2928
Writing CSV…
CSV written to: /workspace/data/duckdb_geojoin_results.csv


In [6]:
print("wxareas columns:")
print(con.execute("PRAGMA table_info('wxareas');").fetchall())

print("\nzipcodes columns:")
print(con.execute("PRAGMA table_info('zipcodes');").fetchall())


wxareas columns:
[(0, 'STATE', 'VARCHAR', False, None, False), (1, 'CWA', 'VARCHAR', False, None, False), (2, 'TIME_ZONE', 'VARCHAR', False, None, False), (3, 'FE_AREA', 'VARCHAR', False, None, False), (4, 'ZONE', 'VARCHAR', False, None, False), (5, 'NAME', 'VARCHAR', False, None, False), (6, 'STATE_ZONE', 'VARCHAR', False, None, False), (7, 'LON', 'DOUBLE', False, None, False), (8, 'LAT', 'DOUBLE', False, None, False), (9, 'SHORTNAME', 'VARCHAR', False, None, False), (10, 'geometry', 'GEOMETRY', False, None, False), (11, '__index_level_0__', 'BIGINT', False, None, False), (12, 'geom_aea', 'GEOMETRY', False, None, False)]

zipcodes columns:
[(0, 'ZCTA5CE20', 'VARCHAR', False, None, False), (1, 'GEOID20', 'VARCHAR', False, None, False), (2, 'CLASSFP20', 'VARCHAR', False, None, False), (3, 'MTFCC20', 'VARCHAR', False, None, False), (4, 'FUNCSTAT20', 'VARCHAR', False, None, False), (5, 'ALAND20', 'BIGINT', False, None, False), (6, 'AWATER20', 'BIGINT', False, None, False), (7, 'INTPTLAT20